In [12]:
import os
from dotenv import load_dotenv

from langgraph.prebuilt import create_react_agent
from langchain.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.utilities import SerpAPIWrapper

In [17]:
load_dotenv()

gemini_api_key = os.getenv('GOOGLE_API_KEY')
serp_api_key = os.getenv("SERP_API_KEY")

os.environ["SERPAPI_API_KEY"] = serp_api_key

In [6]:
llm = ChatGoogleGenerativeAI(
	model="gemini-2.0-flash",
	api_key=gemini_api_key,
	temperature=0.2
)

In [7]:
@tool
def add(x : int , y : int) -> int:
    """Addition of two numbers"""
    return x+y

@tool
def sub(x : int , y : int) -> int:
    """Subtraction of two numbers"""
    return x-y

@tool
def multiple(x : int , y : int) -> int:
    """Multiplication of two numbers"""
    return x*y

@tool
def division(x : int , y : int) -> int:
    """Division of two numbers"""
    return x/y

In [39]:
@tool
def search(query):
    """Searches the web"""
    search = SerpAPIWrapper(search_engine="Google")
    return search.run(query)

In [44]:
prompt = """
You are a helpful Orchestrator Agent. If the user input matches any of the tools call it with appropriate input.
If user asks anything unrelated to the available arithmetic tools, user serpapi to search the web use {user_input} as input for the search tool
If the user greets (e.g: Hi,Hello,Good morning) respond naturally like "Hello, How can I assist you today?" Do not use tools for this."
Only call the tools when absolutely necessary, otherwise respond naturally.
Do NOT make up facts or invent tools. Avoid Hallucinations.
"""

In [45]:
agent_executor = create_react_agent(
	model=llm,
	tools=[add,sub,multiple,division,search],
	prompt=prompt
)

In [46]:
inputs = [
	"add 4 and 7",
	"Who is the prime minister of Nepal?",
	"Multiply 29 and 2",
	"President of USA"
]

In [47]:
# Use the agent


for input in inputs:
	input_message = {
		"role": "user",
		"content": input,
	}
	for step in agent_executor.stream(
		{"messages": [input_message]}, stream_mode="values"
	):
		step["messages"][-1].pretty_print()

================================ Human Message =================================

add 4 and 7
================================== Ai Message ==================================
Tool Calls:
  add (73748865-902d-44bf-b23b-8b60e70fbbb6)
 Call ID: 73748865-902d-44bf-b23b-8b60e70fbbb6
  Args:
    x: 4.0
    y: 7.0
================================= Tool Message =================================
Name: add

11
================================== Ai Message ==================================

The sum of 4 and 7 is 11.
================================ Human Message =================================

Who is the prime minister of Nepal?
================================== Ai Message ==================================
Tool Calls:
  search (3f868473-3431-4bbe-8a1f-9d6abbecf33d)
 Call ID: 3f868473-3431-4bbe-8a1f-9d6abbecf33d
  Args:
    query: Who is the prime minister of Nepal?
================================= Tool Message =================================
Name: search

['KP Sharma Oli type: Prime Mini